# SpaceX Launch Sites — Interactive Folium MapsCreates marker maps and clustered, class-coloured launch records. The 90-launch geographic dataset is loaded from the course URL.

In [ ]:
import pandas as pdimport foliumfrom folium.plugins import MarkerCluster, MousePositionfrom math import radians, sin, cos, asin, sqrturl = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"df = pd.read_csv(url)launch_sites = df.groupby("Launch Site", as_index=False).first()

## Launch-site marker map

In [ ]:
m = folium.Map(location=[35, -100], zoom_start=3)for _, r in launch_sites.iterrows():    pos = [r["Lat"], r["Long"]]    folium.Circle(pos, radius=1000, color="#2563eb", fill=True, fill_opacity=.25).add_to(m)    folium.Marker(pos, popup=r["Launch Site"]).add_to(m)m

## Class-coloured launch outcomes with MarkerCluster

In [ ]:
cluster = MarkerCluster().add_to(m)df["marker_color"] = df["class"].map({1: "green", 0: "red"})for _, r in df.iterrows():    folium.Marker([r["Lat"], r["Long"]],        popup=f'{r["Launch Site"]}: Class={r["class"]}',        icon=folium.Icon(color=r["marker_color"])).add_to(cluster)MousePosition().add_to(m)m

## Great-circle distance to the recorded coastline point

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):    R = 6371.0    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2    return 2*R*asin(sqrt(a))site = launch_sites[launch_sites["Launch Site"].str.contains("SLC 40")].iloc[0]coast = (28.56367, -80.57163)d_km = haversine_km(site["Lat"], site["Long"], coast[0], coast[1])print(f"Coastline distance: {d_km:.2f} km")folium.Marker(coast, popup=f"Coastline: {d_km:.2f} km").add_to(m)folium.PolyLine([[site["Lat"], site["Long"]], coast], color="blue").add_to(m)m